#  JUNCTION TREE VAE - UNDERSTANDING THE APPROACH

---

## WHY JT-VAE IS BETTER:

**Character VAE thinks SMILES is just text:**
- Input: `"CC(=O)O"` → `[C, C, (, =, O, ), O]` → Generate character by character
- **Problem:** Often generates invalid SMILES like `"C(C)=O)"` (unmatched parentheses)

**Junction Tree VAE understands chemistry:**
- Input: `"CC(=O)O"` → Breaks into fragments: `[CH3, C=O, OH]` → Tree structure
- Generates valid fragments → Assembles into valid molecule → **Always works!**

---

## THE KEY IDEA:

**Step 1:** Break molecule into "building blocks" (substructures)
- Example: Aspirin → [benzene ring] + [carboxyl group] + [ester group] + [methyl]

**Step 2:** Create tree showing how blocks connect
- Tree shows: benzene connects to ester, ester connects to carboxyl, etc.

**Step 3:** Encode BOTH the tree structure AND the atoms in each block
- Two encoders: **Tree Encoder** + **Graph Encoder**

**Step 4:** Decode to generate: Tree structure first, then fill in atoms
- **Always produces chemically valid molecules!**

---

##  REQUIRED READING:

**Paper:** "Junction Tree Variational Autoencoder for Molecular Graph Generation"
- **Authors:** Wengong Jin, Regina Barzilay, Tommi Jaakkola (MIT, 2018)
- **Link:** https://arxiv.org/abs/1802.04364

**Read the paper (30-45 min) - focus on:**
- Section 2: Junction Tree Encoder-Decoder
- Figure 2: Architecture diagram
- Algorithm 1: Tree decomposition

---

##  CODE REPOSITORY:

**GitHub:** https://github.com/wengong-jin/icml18-jtnn

**Clone and explore (30-45 min):**
- `fast_jtnn/` - Core implementation
- Look at: `jtnn_enc.py`, `jtnn_dec.py`

---

** Start by reading the paper, then clone the repo!**

# JT-VAE ARCHITECTURE OVERVIEW

#### ENCODING (Molecule → Latent Code): ####

Input: SMILES string "CC(=O)OC1=CC=CC=C1C(=O)O"
   ↓
1. Junction Tree Decomposition
   → Break into molecular fragments (vocab of ~780 common substructures)
   → Create tree showing connections
   
   Example fragments:
   - Benzene ring
   - Carboxyl group (COOH)
   - Ester linkage
   - Methyl group
   
   ↓
2. Tree Encoder (GRU on tree structure)
   → Encodes HOW fragments connect
   → Output: Tree embedding
   
   ↓
3. Graph Encoder (GNN on molecular graph)
   → Encodes atom-level details within each fragment
   → Output: Graph embedding
   
   ↓
4. Combine embeddings
   → Latent code: [tree_embedding + graph_embedding]
   → Typically 56-dimensional vector


DECODING (Latent Code → New Molecule):

Input: Latent code [56 numbers]
   ↓
1. Tree Decoder
   → Generates tree structure (which fragments, how connected)
   → Picks fragments from vocabulary
   
   ↓
2. Graph Decoder
   → Fills in atom details for each fragment
   → Ensures chemical validity
   
   ↓
Output: Valid SMILES string (new molecule!)


KEY COMPONENTS YOU'LL NEED:

1. Vocabulary of molecular fragments (~780 substructures)
2. Tree decomposition algorithm (breaks SMILES → tree)
3. Tree encoder/decoder (GRU-based)
4. Graph encoder/decoder (GNN-based)
5. Loss function (reconstruction + KL divergence + tree/graph losses)

 Now read the paper to understand this deeply!https://arxiv.org/abs/1802.04364

In [ ]:
# Clone JT-VAE repository
!git clone https://github.com/wengong-jin/icml18-jtnn.git

# Install dependencies
!pip install torch rdkit networkx scipy -q

print(" Repository cloned!")
print(" Dependencies installed!")

Cloning into 'icml18-jtnn'...
remote: Enumerating objects: 549, done.
remote: Total 549 (delta 0), reused 0 (delta 0), pack-reused 549 (from 1)
Receiving objects: 100% (549/549), 212.84 MiB | 22.16 MiB/s, done.
Resolving deltas: 100% (220/220), done.
Updating files: 100% (201/201), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 12.1 MB/s eta 0:00:00
 Repository cloned!
 Dependencies installed!


Cell 2: Fix All Python 2→3 Issues

In [ ]:
import os
import re

print(" Converting Python 2 → Python 3...\n")

# Auto-convert with 2to3
!2to3 -w icml18-jtnn/fast_jtnn/ --no-diffs

print("\n Applying additional fixes...\n")

# Get all Python files
py_files = [f'icml18-jtnn/fast_jtnn/{f}' for f in os.listdir('icml18-jtnn/fast_jtnn') if f.endswith('.py')]

for filepath in py_files:
    with open(filepath, 'r') as f:
        content = f.read()

    modified = False

    # Fix cPickle
    if 'cPickle' in content:
        content = content.replace('import cPickle as pickle', 'import pickle')
        content = content.replace('import cPickle', 'import pickle')
        content = content.replace('cPickle', 'pickle')
        modified = True

    # Fix xrange
    if 'xrange' in content:
        content = content.replace('xrange', 'range')
        modified = True

    # Fix iteritems
    if 'iteritems' in content:
        content = content.replace('.iteritems()', '.items()')
        modified = True

    # Fix integer division
    if 'hidden_size / 2' in content or 'latent_size / 2' in content:
        content = content.replace('hidden_size / 2', 'hidden_size // 2')
        content = content.replace('latent_size / 2', 'latent_size // 2')
        modified = True

    if modified:
        with open(filepath, 'w') as f:
            f.write(content)
        print(f"Fixed: {os.path.basename(filepath)}")

print("\n All Python 2→3 conversions complete!")

 Converting Python 2 → Python 3...

/bin/bash: line 1: 2to3: command not found

 Applying additional fixes...

Fixed: chemutils.py
Fixed: jtnn_vae.py
Fixed: mpn.py
Fixed: jtmpn.py
Fixed: jtnn_enc.py
Fixed: jtnn_dec.py
Fixed: datautils.py

 All Python 2→3 conversions complete!


Fix Prints

In [ ]:
import re

print(" Fixing remaining print statements...\n")

py_files = [f'icml18-jtnn/fast_jtnn/{f}' for f in os.listdir('icml18-jtnn/fast_jtnn') if f.endswith('.py')]

for filepath in py_files:
    with open(filepath, 'r') as f:
        lines = f.readlines()

    fixed_lines = []
    modified = False

    for line in lines:
        # Fix old-style print statements: print x → print(x)
        if re.match(r'\s*print\s+[^(]', line) and 'print(' not in line:
            # Extract indentation
            indent = len(line) - len(line.lstrip())
            # Get what comes after 'print '
            rest = line.strip()[6:]  # Remove 'print '
            # Recreate line with print()
            line = ' ' * indent + f'print({rest})\n'
            modified = True

        fixed_lines.append(line)

    if modified:
        with open(filepath, 'w') as f:
            f.writelines(fixed_lines)
        print(f"Fixed: {os.path.basename(filepath)}")

print("\n All print statements fixed!")

🔧 Fixing remaining print statements...

Fixed: chemutils.py
Fixed: jtnn_dec.py
Fixed: mol_tree.py

 All print statements fixed!


Fix mpn.py (map() issue)

In [ ]:
print(" Final clean fix for mpn.py...\n")

# Restore original
!cd icml18-jtnn && git checkout fast_jtnn/mpn.py

# Apply basic 2to3
!2to3 -w icml18-jtnn/fast_jtnn/mpn.py --no-diffs

# Read file
filepath = 'icml18-jtnn/fast_jtnn/mpn.py'
with open(filepath, 'r') as f:
    lines = f.readlines()

# Find and replace ONLY the atom_features function
new_lines = []
skip_lines = 0

for i, line in enumerate(lines):
    if skip_lines > 0:
        skip_lines -= 1
        continue

    if 'def atom_features(atom):' in line:
        # Write the corrected function
        new_lines.append(line)
        new_lines.append('    return torch.Tensor(list(onek_encoding_unk(atom.GetSymbol(), ELEM_LIST)) \n')
        new_lines.append('            + list(onek_encoding_unk(atom.GetDegree(), [0,1,2,3,4,5])) \n')
        new_lines.append('            + list(onek_encoding_unk(atom.GetFormalCharge(), [-1,-2,1,2,0])) \n')
        new_lines.append('            + list(onek_encoding_unk(int(atom.GetChiralTag()), [0,1,2,3])) \n')
        new_lines.append('            + [atom.GetIsAromatic()])\n')
        # Skip next 5 lines (old function body)
        skip_lines = 5
    else:
        new_lines.append(line)

with open(filepath, 'w') as f:
    f.writelines(new_lines)

print(" mpn.py FINALLY fixed correctly!")

 Final clean fix for mpn.py...

Updated 1 path from the index
/bin/bash: line 1: 2to3: command not found
 mpn.py FINALLY fixed correctly!


Load Vocabulary

In [ ]:
import sys
sys.path.insert(0, 'icml18-jtnn/fast_jtnn')

from vocab import Vocab

print(" Loading vocabulary...\n")

# Read vocab file into list
with open('icml18-jtnn/data/zinc/vocab.txt', 'r') as f:
    smiles_list = [line.strip() for line in f.readlines()]

# Create vocabulary
vocab = Vocab(smiles_list)

print(f" Vocabulary loaded!")
print(f"   Total fragments: {vocab.size()}")
print(f"\n Sample fragments:")
for i in range(5):
    print(f"   {i+1}. {vocab.get_smiles(i)}")

 Loading vocabulary...

 Vocabulary loaded!
   Total fragments: 780

 Sample fragments:
   1. C1=[NH+]C=[NH+]CC1
   2. C1=CCCCCC=CCC1
   3. C1CCN[NH2+]CC1
   4. C1CNCCCNC1
   5. C1CCSNC1


Cell 5: Test JT-VAE

In [ ]:
from jtnn_vae import JTNNVAE
from mol_tree import MolTree
from rdkit import Chem
import torch

print(" Testing JT-VAE Pipeline...\n")

# Create model
model = JTNNVAE(vocab, hidden_size=450, latent_size=56, depthT=3, depthG=3)
model.eval()

print(" Model created!")
print(f"   Parameters: ~{sum(p.numel() for p in model.parameters()):,}")

# Test tree decomposition
test_smiles = "CC(=O)Oc1ccccc1C(=O)O"  # Aspirin
print(f"\n Testing molecule: {test_smiles}")

mol_tree = MolTree(test_smiles)

print(f" Junction tree decomposition works!")
print(f"   Total fragments: {len(mol_tree.nodes)}")
print(f"\n Aspirin fragments:")
for i, node in enumerate(mol_tree.nodes):
    print(f"   {i+1}. {node.smiles}")

print(f"\n JT-VAE IS READY FOR TRAINING!")
print(f"\n Next step: Upload your ChEMBL data and train!")

 Testing JT-VAE Pipeline...

 Model created!
   Parameters: ~5,676,743

 Testing molecule: CC(=O)Oc1ccccc1C(=O)O
 Junction tree decomposition works!
   Total fragments: 10

 Aspirin fragments:
   1. CC
   2. C=O
   3. CO
   4. CO
   5. CC
   6. C=O
   7. CO
   8. C1=CC=CC=C1
   9. C
   10. C

 JT-VAE IS READY FOR TRAINING!

 Next step: Upload your ChEMBL data and train!


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:44: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  self.reduction: str = _Reduction.legacy_get_string(size_average, reduce)


In [ ]:
from google.colab import files
import shutil

print(" Saving all your work...\n")

# Create a backup folder
!mkdir -p /content/jtvae_backup

# Copy the entire fixed repo
!cp -r icml18-jtnn /content/jtvae_backup/

# Create a zip file
!cd /content && zip -r jtvae_fixed.zip jtvae_backup/

print(" Everything packaged!\n")

# Download the zip
print(" Downloading jtvae_fixed.zip...")
files.download('/content/jtvae_fixed.zip')

print("\n SAVED!")
print("Next session: Just upload this zip and extract it!")

 Saving all your work...

  adding: jtvae_backup/ (stored 0%)
  adding: jtvae_backup/icml18-jtnn/ (stored 0%)
  adding: jtvae_backup/icml18-jtnn/README.md (deflated 56%)
  adding: jtvae_backup/icml18-jtnn/bo/ (stored 0%)
  adding: jtvae_backup/icml18-jtnn/bo/results1/ (stored 0%)
  adding: jtvae_backup/icml18-jtnn/bo/results1/scores0.dat (stored 0%)
  adding: jtvae_backup/icml18-jtnn/bo/results1/valid_smiles4.dat (stored 0%)
  adding: jtvae_backup/icml18-jtnn/bo/results1/scores4.dat (stored 0%)
  adding: jtvae_backup/icml18-jtnn/bo/results1/valid_smiles1.dat (stored 0%)
  adding: jtvae_backup/icml18-jtnn/bo/results1/valid_smiles2.dat (stored 0%)
  adding: jtvae_backup/icml18-jtnn/bo/results1/valid_smiles3.dat (stored 0%)
  adding: jtvae_backup/icml18-jtnn/bo/results1/valid_smiles0.dat (stored 0%)
  adding: jtvae_backup/icml18-jtnn/bo/results1/scores3.dat (stored 0%)
  adding: jtvae_backup/icml18-jtnn/bo/results1/scores2.dat (stored 0%)
  adding: jtvae_backup/icml18-jtnn/bo/results1/sco

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 SAVED!
Next session: Just upload this zip and extract it!


Verify the CheMBL Parquets load correctly

In [ ]:
import pandas as pd

print(" Verifying ChEMBL data...\n")

# Load and check each file
train = pd.read_parquet('chembl_train.parquet')
val = pd.read_parquet('chembl_val.parquet')
test = pd.read_parquet('chembl_test.parquet')

print(f" Training set: {len(train):,} molecules")
print(f" Validation set: {len(val):,} molecules")
print(f" Test set: {len(test):,} molecules")
print(f"\n Total: {len(train) + len(val) + len(test):,} molecules")

print("\n Sample SMILES from training set:")
for i, smiles in enumerate(train['smiles'].head(5)):
    print(f"   {i+1}. {smiles}")

print("\n ALL DATA READY FOR TRAINING!")

 Verifying ChEMBL data...

 Training set: 34,914 molecules
 Validation set: 7,482 molecules
 Test set: 7,482 molecules

 Total: 49,878 molecules

 Sample SMILES from training set:
   1. O=C1CCN(c2ccccc2)N1
   2. CC(C)Oc1ccccc1OCCCN1CCC(C(O)(c2ccc(F)cc2)c2ccc(F)cc2)CC1
   3. COc1ccc([C@@H]2[C@@H](C(=O)O)[C@H](c3ccc4c(c3)OCO4)CN2CC(=O)N(C)c2ccccc2)cc1
   4. O=C(Nc1ccccc1NC(=O)c1cc(O)c(O)c(O)c1)c1cc(O)c(O)c(O)c1
   5. CC(=O)NCc1ccc(-c2csc(NC(=N)NCC3CCCC3)n2)o1

 ALL DATA READY FOR TRAINING!


In [ ]:
import pandas as pd
from datautils import MolTreeFolder
import torch

print(" Starting JT-VAE Training...\n")

# Load ChEMBL data
train_df = pd.read_parquet('chembl_train.parquet')
print(f" Loaded {len(train_df):,} training molecules")

# Save SMILES to text file (JT-VAE expects text file)
with open('chembl_train.txt', 'w') as f:
    for smiles in train_df['smiles']:
        f.write(smiles + '\n')

print(f" Saved training data to chembl_train.txt")
print(f"\n Ready to train on {len(train_df):,} molecules!")
print(f" Estimated time with Colab Pro: 3-4 hours")

 Starting JT-VAE Training...

 Loaded 34,914 training molecules
 Saved training data to chembl_train.txt

 Ready to train on 34,914 molecules!
 Estimated time with Colab Pro: 3-4 hours


In [ ]:
import os
import pandas as pd

print(" Setting up data folder structure...\n")

# Create data folder
os.makedirs('chembl_data_folder', exist_ok=True)

# Load ChEMBL data
train_df = pd.read_parquet('chembl_train.parquet')
print(f" Loaded {len(train_df):,} training molecules")

# Split into chunks (MolTreeFolder expects multiple files)
chunk_size = 5000
num_chunks = len(train_df) // chunk_size + 1

for i in range(num_chunks):
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, len(train_df))
    chunk_smiles = train_df['smiles'].iloc[start_idx:end_idx]

    # Save chunk
    with open(f'chembl_data_folder/train_{i}.txt', 'w') as f:
        for smiles in chunk_smiles:
            f.write(smiles + '\n')

print(f" Created {num_chunks} data files in chembl_data_folder/")
print(" Files:")
!ls -lh chembl_data_folder/

print("\n Ready to create dataset!")

 Setting up data folder structure...

 Loaded 34,914 training molecules
 Created 7 data files in chembl_data_folder/
 Files:
total 1.5M
-rw-r--r-- 1 root root 214K Mar 14 21:06 train_0.txt
-rw-r--r-- 1 root root 216K Mar 14 21:06 train_1.txt
-rw-r--r-- 1 root root 216K Mar 14 21:06 train_2.txt
-rw-r--r-- 1 root root 217K Mar 14 21:06 train_3.txt
-rw-r--r-- 1 root root 214K Mar 14 21:06 train_4.txt
-rw-r--r-- 1 root root 216K Mar 14 21:06 train_5.txt
-rw-r--r-- 1 root root 211K Mar 14 21:06 train_6.txt

 Ready to create dataset!


NEW Day Reload | Unzip File

In [ ]:
!unzip -q jtvae_fixed.zip
!mv jtvae_backup/icml18-jtnn .
!rm -rf jtvae_backup

print(" JT-VAE code extracted!")

mv: cannot move 'jtvae_backup/icml18-jtnn' to './icml18-jtnn': Directory not empty
 JT-VAE code extracted!


Install Packages

In [ ]:
!pip install torch rdkit networkx scipy pandas -q
print(" Packages installed!")

 Packages installed!


Load Vocabulary

In [ ]:
import sys
sys.path.insert(0, 'icml18-jtnn/fast_jtnn')
from vocab import Vocab

# Load vocab
with open('icml18-jtnn/data/zinc/vocab.txt', 'r') as f:
    smiles_list = [line.strip() for line in f.readlines()]

vocab = Vocab(smiles_list)
print(f" Vocab loaded: {vocab.size()} fragments")

 Vocab loaded: 780 fragments


Creating Data Folder

In [ ]:
import os
import pandas as pd

os.makedirs('chembl_data_folder', exist_ok=True)

# Load and split data
train_df = pd.read_parquet('chembl_train.parquet')
print(f" Loaded {len(train_df):,} molecules")

# Split into chunks
chunk_size = 5000
num_chunks = len(train_df) // chunk_size + 1

for i in range(num_chunks):
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, len(train_df))
    chunk_smiles = train_df['smiles'].iloc[start_idx:end_idx]

    with open(f'chembl_data_folder/train_{i}.txt', 'w') as f:
        for smiles in chunk_smiles:
            f.write(smiles + '\n')

print(f" Created {num_chunks} files in chembl_data_folder/")

 Loaded 34,914 molecules
 Created 7 files in chembl_data_folder/


MolTreeFolder Expects Preprocessed Pickle Files!
We need to preprocess the SMILES first!

In [ ]:
from mol_tree import MolTree
import pickle
import os

print(" Preprocessing SMILES into MolTree format...\n")

# Process each data file
data_folder = 'chembl_data_folder'
files = [f for f in os.listdir(data_folder) if f.endswith('.txt')]

for txt_file in files:
    print(f"Processing {txt_file}...")

    # Read SMILES
    with open(os.path.join(data_folder, txt_file), 'r') as f:
        smiles_list = [line.strip() for line in f.readlines()]

    # Convert to MolTree objects
    mol_trees = []
    for i, smiles in enumerate(smiles_list):
        try:
            mol_tree = MolTree(smiles)
            mol_trees.append(mol_tree)
        except:
            continue  # Skip invalid SMILES

        if (i + 1) % 1000 == 0:
            print(f"  Processed {i+1}/{len(smiles_list)} molecules...")

    # Save as pickle
    pkl_file = txt_file.replace('.txt', '.pkl')
    with open(os.path.join(data_folder, pkl_file), 'wb') as f:
        pickle.dump(mol_trees, f)

    # Remove txt file
    os.remove(os.path.join(data_folder, txt_file))

    print(f" Saved {len(mol_trees)} molecules to {pkl_file}\n")

print(" Preprocessing complete!")

 Preprocessing SMILES into MolTree format...

Processing train_2.txt...


[23:22:54] Can't kekulize mol.  Unkekulized atoms: 7 8 9 10 11
[23:22:55] Can't kekulize mol.  Unkekulized atoms: 2 3 4 18 19
[23:22:55] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 7
[23:22:55] Can't kekulize mol.  Unkekulized atoms: 12 13 18 19 25
[23:22:55] Can't kekulize mol.  Unkekulized atoms: 12
[23:22:55] Can't kekulize mol.  Unkekulized atoms: 4 5 6
[23:22:55] Can't kekulize mol.  Unkekulized atoms: 10 16 22
[23:22:55] Can't kekulize mol.  Unkekulized atoms: 4
[23:22:55] Can't kekulize mol.  Unkekulized atoms: 4 5 6 7 8
[23:22:55] Can't kekulize mol.  Unkekulized atoms: 16
[23:22:55] Can't kekulize mol.  Unkekulized atoms: 5 7 8 9 13
[23:22:56] Can't kekulize mol.  Unkekulized atoms: 2 4 5 6 22
[23:22:56] Can't kekulize mol.  Unkekulized atoms: 1 2 4 13 26
[23:22:56] Can't kekulize mol.  Unkekulized atoms: 6 18 23
[23:22:56] Can't kekulize mol.  Unkekulized atoms: 7 8 9
[23:22:56] Can't kekulize mol.  Unkekulized atoms: 5
[23:22:56] Can't kekulize mol.  Unkekulized atoms: 7

  Processed 1000/5000 molecules...


[23:23:00] Can't kekulize mol.  Unkekulized atoms: 4 11 12
[23:23:00] Can't kekulize mol.  Unkekulized atoms: 9 10 11
[23:23:00] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5
[23:23:00] Can't kekulize mol.  Unkekulized atoms: 7 8 21
[23:23:00] Can't kekulize mol.  Unkekulized atoms: 3 4 5
[23:23:00] Can't kekulize mol.  Unkekulized atoms: 2 7 21
[23:23:00] Can't kekulize mol.  Unkekulized atoms: 9 20 21
[23:23:00] Can't kekulize mol.  Unkekulized atoms: 11
[23:23:01] Can't kekulize mol.  Unkekulized atoms: 11 16 29
[23:23:01] Can't kekulize mol.  Unkekulized atoms: 2 12 17
[23:23:01] Can't kekulize mol.  Unkekulized atoms: 4 5 6 9 10
[23:23:01] Can't kekulize mol.  Unkekulized atoms: 10
[23:23:01] Can't kekulize mol.  Unkekulized atoms: 3 5 13
[23:23:01] Can't kekulize mol.  Unkekulized atoms: 14 21 22
[23:23:01] Can't kekulize mol.  Unkekulized atoms: 12 17 22
[23:23:01] Can't kekulize mol.  Unkekulized atoms: 6
[23:23:01] Can't kekulize mol.  Unkekulized atoms: 1 2 4 12 21
[23:23

  Processed 2000/5000 molecules...


...

  Processed 3000/5000 molecules...


[23:26:57] Can't kekulize mol.  Unkekulized atoms: 4 6 11 12 13
[23:26:57] Can't kekulize mol.  Unkekulized atoms: 2 4 6 7 11
[23:26:57] Can't kekulize mol.  Unkekulized atoms: 15 20 25
[23:26:57] Can't kekulize mol.  Unkekulized atoms: 28
[23:26:57] Can't kekulize mol.  Unkekulized atoms: 11 15 20
[23:26:57] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 14
[23:26:57] Can't kekulize mol.  Unkekulized atoms: 10 11 12 17 18
[23:26:57] Can't kekulize mol.  Unkekulized atoms: 11 14 19
[23:26:57] Can't kekulize mol.  Unkekulized atoms: 9
[23:26:58] Can't kekulize mol.  Unkekulized atoms: 2 7 8 27 28
[23:26:58] Can't kekulize mol.  Unkekulized atoms: 8 9 10 11 13
[23:26:58] Can't kekulize mol.  Unkekulized atoms: 21 22 23 28 29
[23:26:58] Can't kekulize mol.  Unkekulized atoms: 9 10 22 23 28
[23:26:58] Can't kekulize mol.  Unkekulized atoms: 7 13 19
[23:26:58] Can't kekulize mol.  Unkekulized atoms: 16
[23:26:58] Can't kekulize mol.  Unkekulized atoms: 9 10 11 12 13
[23:26:59] Can't kekuli

  Processed 4000/5000 molecules...


[23:27:04] Can't kekulize mol.  Unkekulized atoms: 7 8 10 11 12
[23:27:04] Can't kekulize mol.  Unkekulized atoms: 12
[23:27:04] Can't kekulize mol.  Unkekulized atoms: 10 11 12 17 18
[23:27:04] Can't kekulize mol.  Unkekulized atoms: 10
[23:27:04] Can't kekulize mol.  Unkekulized atoms: 8 20 29
[23:27:04] Can't kekulize mol.  Unkekulized atoms: 9 22 28
[23:27:04] Can't kekulize mol.  Unkekulized atoms: 2 4 5 6 10
[23:27:05] Can't kekulize mol.  Unkekulized atoms: 5 6 7 18 19
[23:27:05] Can't kekulize mol.  Unkekulized atoms: 1 2 4 12 20
[23:27:05] Can't kekulize mol.  Unkekulized atoms: 10 11 12 13 14
[23:27:05] Can't kekulize mol.  Unkekulized atoms: 8
[23:27:05] Can't kekulize mol.  Unkekulized atoms: 15
[23:27:05] Can't kekulize mol.  Unkekulized atoms: 9
[23:27:05] Can't kekulize mol.  Unkekulized atoms: 21 22 23 28 29
[23:27:05] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5
[23:27:05] Can't kekulize mol.  Unkekulized atoms: 15 20 25
[23:27:05] Can't kekulize mol.  Unkekulized

  Processed 5000/5000 molecules...
 Saved 4712 molecules to train_5.pkl

 Preprocessing complete!


Fixing datautils.py for pickle files (From text to binary)

In [ ]:
print(" Fixing datautils.py for pickle files...\n")

filepath = 'icml18-jtnn/fast_jtnn/datautils.py'

with open(filepath, 'r') as f:
    content = f.read()

# Fix: open pickle files in binary mode
content = content.replace(
    "with open(fn) as f:\n                data = pickle.load(f)",
    "with open(fn, 'rb') as f:\n                data = pickle.load(f)"
)

with open(filepath, 'w') as f:
    f.write(content)

print(" Fixed datautils.py!")
print("Now restart runtime and re-run setup cells + training!")

 Fixing datautils.py for pickle files...

 Fixed datautils.py!
Now restart runtime and re-run setup cells + training!


Building custom vocabulary from ChEMBL data

In [ ]:
from mol_tree import MolTree
from vocab import Vocab
import pickle

print(" Building custom vocabulary from ChEMBL data...\n")

# Collect all fragments from training data
all_fragments = set()
processed = 0

for pkl_file in os.listdir('chembl_data_folder'):
    if not pkl_file.endswith('.pkl'):
        continue

    print(f"Extracting fragments from {pkl_file}...")

    with open(f'chembl_data_folder/{pkl_file}', 'rb') as f:
        mol_trees = pickle.load(f)

    for mol_tree in mol_trees:
        for node in mol_tree.nodes:
            all_fragments.add(node.smiles)
        processed += 1

    print(f"  Processed {processed} molecules, {len(all_fragments)} unique fragments")

print(f"\n Found {len(all_fragments)} unique molecular fragments")

# Create new vocabulary
fragment_list = sorted(list(all_fragments))
chembl_vocab = Vocab(fragment_list)

print(f" ChEMBL vocabulary created: {chembl_vocab.size()} fragments")

# Save it
with open('chembl_vocab.pkl', 'wb') as f:
    pickle.dump(chembl_vocab, f)

print(" Saved to chembl_vocab.pkl")

# Update global vocab variable
vocab = chembl_vocab
print("\n Ready to train with ChEMBL vocabulary!")

 Building custom vocabulary from ChEMBL data...

Extracting fragments from train_1.pkl...
  Processed 4711 molecules, 335 unique fragments
Extracting fragments from train_0.pkl...
  Processed 9404 molecules, 394 unique fragments
Extracting fragments from train_2.pkl...
  Processed 14092 molecules, 435 unique fragments
Extracting fragments from train_3.pkl...
  Processed 18800 molecules, 475 unique fragments
Extracting fragments from train_4.pkl...
  Processed 23513 molecules, 496 unique fragments
Extracting fragments from train_6.pkl...
  Processed 28147 molecules, 515 unique fragments
Extracting fragments from train_5.pkl...
  Processed 32859 molecules, 534 unique fragments

 Found 534 unique molecular fragments
 ChEMBL vocabulary created: 534 fragments
 Saved to chembl_vocab.pkl

 Ready to train with ChEMBL vocabulary!


Step 3: Training Cell

In [ ]:
from datautils import MolTreeFolder
from jtnn_vae import JTNNVAE
import torch
import time

print(" Starting JT-VAE Training...\n")

batch_size = 32
learning_rate = 0.001
num_epochs = 10

# MolTreeFolder IS the dataset - don't wrap in DataLoader!
dataset = MolTreeFolder('chembl_data_folder', vocab, batch_size=batch_size)

model = JTNNVAE(vocab, hidden_size=450, latent_size=56, depthT=3, depthG=3)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f" Model on: {device}")
print(f" Dataset ready\n")

start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    batch_count = 0

    # Iterate directly over dataset
    for batch in dataset:
        optimizer.zero_grad()

        loss, kl_div, wacc, tacc, sacc = model(batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        batch_count += 1

        if batch_count % 100 == 0:
            elapsed = (time.time() - start_time) / 60
            print(f"Epoch {epoch+1}/{num_epochs} | Batch {batch_count} | "
                  f"Loss: {loss.item():.4f} | KL: {kl_div:.4f} | Time: {elapsed:.1f}m")

    avg_loss = epoch_loss / batch_count
    print(f"\n Epoch {epoch+1} Complete | Avg Loss: {avg_loss:.4f}")
    print("=" * 60 + "\n")

    if (epoch + 1) % 2 == 0:
        torch.save(model.state_dict(), f'jtvae_epoch_{epoch+1}.pt')
        print(f" Checkpoint saved: jtvae_epoch_{epoch+1}.pt\n")

# Save final model
torch.save(model.state_dict(), 'jtvae_final.pt')

total_time = (time.time() - start_time) / 3600
print(f"\n TRAINING COMPLETE!")
print(f"  Total time: {total_time:.2f} hours")
print(f" Model saved: jtvae_final.pt")

 Starting JT-VAE Training...

 Model on: cpu
 Dataset ready



/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:44: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  self.reduction: str = _Reduction.legacy_get_string(size_average, reduce)


TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/content/icml18-jtnn/fast_jtnn/datautils.py", line 106, in __getitem__
    return tensorize(self.data[idx], self.vocab, assm=self.assm)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/icml18-jtnn/fast_jtnn/datautils.py", line 113, in tensorize
    mpn_holder = MPN.tensorize(smiles_batch)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/icml18-jtnn/fast_jtnn/mpn.py", line 97, in tensorize
    fbonds.append( torch.cat([fatoms[x], bon